# Using Nomad to get DFT metadata

This is the expected process:

1. use nomad parse to get the metadata from the DFT file(s), save as json
2. build json LD file with nomad metadata plus domain specific data (experiment, sample)
3. pack the dft files, the nomad file and the json-ld metadata (Webby/RO-Crate)
4. publish

Can we do this simple workflow for different types of DFT calculations (ORCA, VASP, Q, Gaussian)?

Get files and test


## Example 1 Orca file from Nomad DB
- downloaded the file from nomad [16459](https://nomad-lab.eu/projects/cJ6dPO0ZSHuFiwQIip8tLg/files/16459)
- created scritpt to parse and save metadata as JSON
- created script to add domain metadata and save as JSON-LD
- created script to save all as RO-Crate

Repeat process for other similar files

We are supporting:
1. reproducibility of results
2. increase reuse of data
3. interoperability

Are these actions enough to say we are producing AI and ML ready data?


## Parse DFT data using NOMAD

The functions below will parse the ORCA data and produce a JSON file with NOMAD metadata

In [1]:
import sys
import json
from nomad.client import parse, normalize_all

# parse the input file
def parse_file(archive_in):
    # Match and run the parser
    archive = parse(archive_in) #parse(sys.argv[1])

    # Run all normalizers
    normalize_all(archive[0])

    # Convert to a Python dictionary
    python_dict = archive[0].run[0].m_to_dict()

    return python_dict

#save the parsed json metadata
def save_parsed(python_dict, archive_out):
    # save
    with open(archive_out, "w") as f:
        json.dump(python_dict, f, indent=2)

/home/abraham/venvs/nomad/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dft_file_input = "./inputs/nomad_example_01_orcafiles/16459/16459.out"
json_nomad_file = "./outputs/orca_examples/16459/16459.nomad.json"

parsed_data = parse_file(dft_file_input)
save_parsed(parsed_data, json_nomad_file)

parsed_data

{'program': {'name': 'ORCA', 'version': '3.0.3 - RELEASE   -'},
 'method': [{'dft': {'xc_functional': {'hybrid': [{'name': 'HYB_GGA_XC_PBEH'}]}},
   'electronic': {'method': 'DFT'},
   'scf': {'native_tier': 'TightSCF',
    'threshold_energy_change': 4.3597447222071695e-26,
    'threshold_density_change': 5e-09},
   'electrons_representation': [{'type': 'atom-centered orbitals',
     'scope': ['auxiliary'],
     'basis_set': [{'type': 'gaussians'},
      {'type': 'gaussians'},
      {'type': 'gaussians'},
      {'type': 'gaussians'}]}]},
  {'starting_method_ref': '#/run/0/method/0',
   'methods_ref': ['#/run/0/method/0'],
   'electronic': {'method': 'TDDFT'}}],
 'system': [{'atoms': {'labels': ['N',
     'C',
     'C',
     'C',
     'O',
     'C',
     'O',
     'C',
     'H',
     'H',
     'H',
     'H',
     'H'],
    'positions': [[7.5683e-12, 1.3470580000000002e-10, 9.5055e-12],
     [-3.7305e-12, -3.127e-13, 3.7585e-12],
     [-1.153743e-10, -7.772210000000001e-11, -2.578e-12],


## Create JSON-LD file with context metadata
The add context fucntion takes the json file and adds context metadata that is then saved as JSON-LD

This is where we would map the NOMAD json metadata with Croisant metadata. So a similar operation to add_context but in this case call it map_to_croissant or something similar

We should preserve the relevant context data (system, material, etc.) to keep it human readable as well as AI ready.


In [3]:
import json

def add_context(json_file, json_ld_out):
    # 1. Load your standard NOMAD parse output
    with open(json_file, "r") as f:
        nomad_data = json.load(f)
    
    # Extract core information from the NOMAD structure
    calc_block = nomad_data.get("calculation", [{}])[-1]
    system_block = nomad_data.get("system", [{}])[-1]
    
    # 2. Build the JSON-LD structure mapping NOMAD to Semantic Vocabularies
    json_ld_archive = {
        # Establish ontologies for both general metadata and chemical specifics
        "@context": {
            "@vocab": "https://schema.org",
            "chmo": "http://obolibrary.org",  # Chemical Methods Ontology
            "chebi": "http://obolibrary.org",  # Chemical Entities of Biological Interest
            "nomad": "https://nomad-lab.eu",
            "energy": "nomad:run/calculation/energy/total",
            "positions": "nomad:run/system/atoms/positions"
        },
        "@id": "https://doi.org",
        "@type": ["Dataset", "chmo:CHMO_0001402"],  # Dataset + Density Functional Theory Calculation type
        "name": "DFT Simulation of Molecular Catalyst Sample",
        
        # Inject your highly detailed custom sample definitions natively into the schema
        "about": {
            "@type": "chebi:CHEBI_23367",  # Molecular entity identifier
            "name": "Iron-based Coordination Catalyst",
            "description": "Fe(II) complex with a tetradentate nitrogen-donor macrocyclic ligand.",
            # the formula should be obtainable from NOMAD, no need for asking the user again
            "chemicalFormula": "C20H24FeN4",
            "additionalType": "https://example.org"
        },
        
        # Map the standardized calculations exported by NOMAD
        "variableMeasured": [
            {
                "@type": "PropertyValue",
                "name": "Total Electronic Energy",
                "propertyID": "energy",
                "value": calc_block.get("energy", {}).get("total", {}).get("value"),
                "unitText": "eV"
            },
            {
                "@type": "PropertyValue",
                "name": "Atomic Positions",
                "propertyID": "positions",
                "value": system_block.get("atoms", {}).get("positions")
            }
        ],
        
        # Preserve provenance software markers
        "creator": {
            "@type": "SoftwareApplication",
            "name": nomad_data.get("program", {}).get("name"),
            "softwareVersion": nomad_data.get("program", {}).get("version")
        }
    }
    
    # 3. Save as a dedicated .jsonld file
    with open(json_ld_out, "w") as f:
        json.dump(json_ld_archive, f, indent=2)
    return json_ld_archive

In [4]:
json_ld_file = "./outputs/orca_examples/16459/16459_nomad_orca.jsonld"

json_ld_out = add_context(json_nomad_file, json_ld_file)

In [5]:
json_ld_out

{'@context': {'@vocab': 'https://schema.org',
  'chmo': 'http://obolibrary.org',
  'chebi': 'http://obolibrary.org',
  'nomad': 'https://nomad-lab.eu',
  'energy': 'nomad:run/calculation/energy/total',
  'positions': 'nomad:run/system/atoms/positions'},
 '@id': 'https://doi.org',
 '@type': ['Dataset', 'chmo:CHMO_0001402'],
 'name': 'DFT Simulation of Molecular Catalyst Sample',
 'about': {'@type': 'chebi:CHEBI_23367',
  'name': 'Iron-based Coordination Catalyst',
  'description': 'Fe(II) complex with a tetradentate nitrogen-donor macrocyclic ligand.',
  'chemicalFormula': 'C20H24FeN4',
  'additionalType': 'https://example.org'},
 'variableMeasured': [{'@type': 'PropertyValue',
   'name': 'Total Electronic Energy',
   'propertyID': 'energy',
   'value': -1.7368833470628061e-15,
   'unitText': 'eV'},
  {'@type': 'PropertyValue',
   'name': 'Atomic Positions',
   'propertyID': 'positions',
   'value': [[7.5683e-12, 1.3470580000000002e-10, 9.5055e-12],
    [-3.7305e-12, -3.127e-13, 3.7585e

## Create package ready for publishing

get all input files and the associated json metadata in a single structure ready for publishing.

Could be something like an RO-Crate or similar.
